In [1]:
import sys
from pathlib import Path

ROOT = Path('/home/aceya/EMS')
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

from ems.db import load_env, connect

load_env()

In [2]:
# 대상 계량기 정의
# H1.K12: 2022년 이후 P/qv 0 고착, Tdiff 노이즈 극심 -> 제외
# H1.K15: 설비 미가동 상태, 간헐적 스파이크만 존재 -> 제외

COOLING_ELECTRIC = ['H1.Z16', 'H1.Z11', 'H1.Z12', 'H1.Z24', 'H1.Z25']
COOLING_THERMAL  = ['V.K21', 'H1.K11', 'H1.K14', 'H1.K16', 'H2.K21']
HEATING_ELECTRIC = ['H1.Z20', 'H1.ZE20']
HEATING_THERMAL  = ['H1.W11', 'H1.W12']

ALL_METERS = COOLING_ELECTRIC + COOLING_THERMAL + HEATING_ELECTRIC + HEATING_THERMAL

# 조회 기간: 학습 구간 2018~2021년
START = '2018-01-01'
END   = '2022-01-01'

print(f'전체 계량기 수: {len(ALL_METERS)}')
print(f'조회 기간: {START} ~ {END}')

전체 계량기 수: 14
조회 기간: 2018-01-01 ~ 2022-01-01


In [4]:
def fetch_meter_data(meter_urn: str) -> pd.DataFrame:
    sql = """
        SELECT
            ts,
            measurement,
            value
        FROM ems.cr_measurement_1h
        WHERE meter_urn = %s
          AND ts >= %s
          AND ts <  %s
        ORDER BY ts, measurement
    """
    with connect() as conn:
        df = pd.read_sql(sql, conn, params=(meter_urn, START, END))

    if df.empty:
        return pd.DataFrame()

    df = df.pivot(index='ts', columns='measurement', values='value')
    df.index = pd.to_datetime(df.index, utc=True).tz_convert('Europe/Berlin')
    df.index.name = 'timestamp'
    return df

In [5]:
# Step 1: 계량기별 상관계수 CSV 저장
save_dir_csv = ROOT / 'outputs/tables/correlation_eda'
os.makedirs(save_dir_csv, exist_ok=True)

for meter in ALL_METERS:
    df = fetch_meter_data(meter)
    if df.empty:
        print(f'{meter} 데이터 없음')
        continue

    # 상관계수 계산 (결측 제거)
    corr = df.corr(method='pearson', numeric_only=True)
    corr.to_csv(save_dir_csv / f'{meter}_corr.csv')
    print(f'{meter} 저장 완료')

/tmp/ipykernel_76602/1051648224.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, START, END))


H1.Z16 저장 완료


/tmp/ipykernel_76602/1051648224.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, START, END))


H1.Z11 저장 완료


/tmp/ipykernel_76602/1051648224.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, START, END))


...

H1.Z20 저장 완료
H1.ZE20 데이터 없음


/tmp/ipykernel_76602/1051648224.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, START, END))
/tmp/ipykernel_76602/1051648224.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, START, END))


H1.W11 저장 완료


/tmp/ipykernel_76602/1051648224.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, START, END))


H1.W12 저장 완료


In [6]:
# Step 2: 히트맵 PNG 저장
save_dir_png = ROOT / 'outputs/figures/correlation_eda'
os.makedirs(save_dir_png, exist_ok=True)

for meter in ALL_METERS:
    csv_path = save_dir_csv / f'{meter}_corr.csv'
    if not csv_path.exists():
        print(f'{meter} CSV 없음')
        continue

    corr = pd.read_csv(csv_path, index_col=0)

    fig = go.Figure(data=go.Heatmap(
        z=corr.values,
        x=corr.columns.tolist(),
        y=corr.index.tolist(),
        colorscale='RdBu',
        zmid=0,
        zmin=-1,
        zmax=1,
        text=corr.round(2).values,
        texttemplate='%{text}',
        showscale=True,
    ))

    fig.update_layout(
        title=f'{meter} 측정항목 간 상관계수 (2018~2021)',
        width=900,
        height=800,
        template='plotly_white',
    )

    fig.write_image(
        str(save_dir_png / f'{meter}_corr_heatmap.png'),
        width=900, height=800
    )
    print(f'{meter} 저장 완료')

H1.Z16 저장 완료
H1.Z11 저장 완료
H1.Z12 저장 완료
H1.Z24 저장 완료
H1.Z25 저장 완료
V.K21 저장 완료
H1.K11 저장 완료
H1.K14 저장 완료
H1.K16 저장 완료
H2.K21 저장 완료
H1.Z20 저장 완료
H1.ZE20 CSV 없음
H1.W11 저장 완료
H1.W12 저장 완료


In [7]:
# Step 3: 고상관 항목 쌍 추출 (|r| >= 0.9)
save_dir_csv2 = ROOT / 'outputs/tables/correlation_eda'
high_corr_list = []

for meter in ALL_METERS:
    csv_path = save_dir_csv2 / f'{meter}_corr.csv'
    if not csv_path.exists():
        continue

    corr = pd.read_csv(csv_path, index_col=0)

    # 상삼각 행렬만 추출 (중복 제거)
    for i in range(len(corr.columns)):
        for j in range(i+1, len(corr.columns)):
            val = corr.iloc[i, j]
            if abs(val) >= 0.9:
                high_corr_list.append({
                    'meter': meter,
                    'measurement_A': corr.columns[i],
                    'measurement_B': corr.columns[j],
                    'correlation': round(val, 4)
                })

high_corr_df = pd.DataFrame(high_corr_list)
high_corr_df = high_corr_df.sort_values(['meter', 'correlation'], ascending=[True, False])
high_corr_df.to_csv(save_dir_csv2 / 'high_correlation_pairs.csv', index=False)
print(f'고상관 항목 쌍 총 {len(high_corr_df)}개')
print(high_corr_df.head(20))

고상관 항목 쌍 총 321개
      meter measurement_A measurement_B  correlation
229  H1.K11             V             W       0.9860
231  H1.K14             V             W       0.9881
230  H1.K14           Trl           Tvl       0.9228
232  H1.K16             V             W       0.9902
316  H1.W11             V             W       0.9975
315  H1.W11             P         Tdiff       0.9689
320  H1.W12             V             W       0.9996
317  H1.W12           Trl           Tvl       0.9831
319  H1.W12           Tvl            qv       0.9438
318  H1.W12           Trl            qv       0.9398
105  H1.Z11             W          W_in       1.0000
106  H1.Z11            WQ         WQ_in       1.0000
87   H1.Z11            P1            P2       0.9997
103  H1.Z11             W            WQ       0.9993
104  H1.Z11             W         WQ_in       0.9993
107  H1.Z11            WQ          W_in       0.9993
108  H1.Z11         WQ_in          W_in       0.9993
61   H1.Z11            I1     